# 📊 Phase 2: Exploratory Data Analysis (EDA)
## Credit Risk Analytics — Home Credit Default Risk
---
**Goal:** Uncover default risk patterns across age, income, loan type, and key financial ratios through visualizations and statistical summaries.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

df = pd.read_csv('application_clean.csv')
print(f"Dataset loaded: {df.shape}")


## 1. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['TARGET'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','crimson'], edgecolor='black')
axes[0].set_title('Target Distribution (0=Repaid, 1=Defaulted)')
axes[0].set_xticklabels(['Repaid', 'Defaulted'], rotation=0)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(df['TARGET'].value_counts(), labels=['Repaid (91.93%)', 'Defaulted (8.07%)'],
            colors=['steelblue','crimson'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Default Rate')

plt.tight_layout()
plt.savefig('plot_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Default rate: {df['TARGET'].mean()*100:.2f}%")


## 2. Default Rate by Age Group

In [ ]:
age_default = df.groupby('AGE_GROUP')['TARGET'].agg(['mean','count']).reset_index()
age_default.columns = ['AGE_GROUP', 'default_rate', 'count']
age_default['default_rate_pct'] = (age_default['default_rate'] * 100).round(2)
age_default = age_default.sort_values('default_rate_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(age_default['AGE_GROUP'], age_default['default_rate_pct'], color='#E07B00', edgecolor='black')
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set_xlabel('Default Rate (%)')
ax.set_title('Default Rate by Age Group')
plt.tight_layout()
plt.savefig('plot_default_by_age.png', dpi=150, bbox_inches='tight')
plt.show()
print(age_default[['AGE_GROUP','default_rate_pct','count']])


## 3. Default Rate by Income Bracket

In [ ]:
income_default = df.groupby('INCOME_BRACKET')['TARGET'].agg(['mean','count']).reset_index()
income_default.columns = ['INCOME_BRACKET', 'default_rate', 'count']
income_default['default_rate_pct'] = (income_default['default_rate'] * 100).round(2)
income_default = income_default.sort_values('default_rate_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(income_default['INCOME_BRACKET'], income_default['default_rate_pct'],
               color='crimson', edgecolor='black')
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set_xlabel('Default Rate (%)')
ax.set_title('Default Rate by Income Bracket')
plt.tight_layout()
plt.savefig('plot_default_by_income.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Default Rate by Loan Type

In [ ]:
loan_default = df.groupby('NAME_CONTRACT_TYPE')['TARGET'].mean().reset_index()
loan_default['default_rate_pct'] = (loan_default['TARGET'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(loan_default['NAME_CONTRACT_TYPE'], loan_default['default_rate_pct'],
            color=['steelblue','crimson'], edgecolor='black')
axes[0].set_title('Default Rate by Loan Type')
axes[0].set_ylabel('Default Rate (%)')

# Count by loan type
loan_count = df['NAME_CONTRACT_TYPE'].value_counts()
axes[1].pie(loan_count, labels=loan_count.index, autopct='%1.1f%%',
            colors=['steelblue','crimson'], startangle=90)
axes[1].set_title('Portfolio Exposure by Loan Type')

plt.tight_layout()
plt.savefig('plot_loan_type.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Credit Amount Distribution by Default Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(['AMT_CREDIT', 'AMT_INCOME_TOTAL']):
    for target, label, color in [(0, 'Repaid', 'steelblue'), (1, 'Defaulted', 'crimson')]:
        axes[i].hist(df[df['TARGET']==target][col], bins=50, alpha=0.6,
                     label=label, color=color, density=True)
    axes[i].set_title(f'{col} Distribution by Default Status')
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.tight_layout()
plt.savefig('plot_amount_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Correlation Heatmap

In [ ]:
# Select key numeric features
key_cols = ['TARGET', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY',
            'AGE', 'ANNUITY_INCOME_RATIO', 'CREDIT_INCOME_RATIO',
            'DAYS_EMPLOYED', 'DAYS_REGISTRATION']

corr_cols = [c for c in key_cols if c in df.columns]
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Key Features')
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. EDA Summary — Key Insights

In [ ]:
print("=" * 55)
print("KEY EDA INSIGHTS — Credit Risk Analytics")
print("=" * 55)

default_rate = df['TARGET'].mean() * 100
print(f"\n1. Overall default rate: {default_rate:.2f}%")

young = df[df['AGE_GROUP'] == '<25']['TARGET'].mean() * 100
old = df[df['AGE_GROUP'] == '55+']['TARGET'].mean() * 100
print(f"\n2. Age risk gap:")
print(f"   Under-25 default rate: {young:.2f}%")
print(f"   55+ default rate: {old:.2f}%")

low_income = df[df['INCOME_BRACKET'] == '<90K']['TARGET'].mean() * 100
high_income = df[df['INCOME_BRACKET'] == '270K-400K']['TARGET'].mean() * 100
print(f"\n3. Income risk gap:")
print(f"   <90K default rate: {low_income:.2f}%")
print(f"   270K-400K default rate: {high_income:.2f}%")

cash = df[df['NAME_CONTRACT_TYPE'] == 'Cash Loans']['TARGET'].mean() * 100
revolving = df[df['NAME_CONTRACT_TYPE'] == 'Revolving Loans']['TARGET'].mean() * 100
print(f"\n4. Loan type default rates:")
print(f"   Cash Loans: {cash:.2f}%")
print(f"   Revolving Loans: {revolving:.2f}%")
print("=" * 55)
